# Methods: Train and Tune

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## Get back to where we were in EDA

In [2]:
# Set the path to the file you'd like to load
file_path = "student_performance_finalscore.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "sarveshchhetri/student-lifestyle-vs-academic-performance-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

In [3]:
categorical_vars = df.select_dtypes(include=['str']).columns.tolist()
df[categorical_vars] = df[categorical_vars].astype("category")

In [4]:
# sanity check
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Student_ID                  8000 non-null   category
 1   Age                         8000 non-null   int64   
 2   Gender                      8000 non-null   category
 3   Hours_Studied               8000 non-null   float64 
 4   Attendance                  8000 non-null   float64 
 5   Sleep_Hours                 8000 non-null   float64 
 6   Stress_Level                8000 non-null   float64 
 7   Screen_Time                 8000 non-null   float64 
 8   Previous_GPA                8000 non-null   float64 
 9   Part_Time_Job               8000 non-null   category
 10  Study_Method                8000 non-null   category
 11  Diet_Quality                8000 non-null   category
 12  Internet_Quality            8000 non-null   category
 13  Extracurricular             8

## Lets split the data

In [5]:
# now lets split up the dataset into X and y
# also remember that we dont want Age, Student_ID, or Final_Score in X

cols_to_drop = ["Student_ID", "Age"]
target = ["Final_Score"]

X = df.drop(columns=cols_to_drop + target)
y = df[target]


In [6]:
# Split the data into train, tune, and test sets
# 70/15/15 split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, random_state=42, test_size=0.30)
X_train, X_tune, y_train, y_tune = train_test_split(X_temp, y_temp, random_state=42, test_size=0.5)

In [ ]:
## PIPELINE STRUCTURE

# define column groups from X (already has dropped columns)
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='category').columns.tolist()

# build pipeline
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# fit on train
pipeline.fit(X_train, y_train)

# evaluate on tune set
y_tune_pred = pipeline.predict(X_tune)
print(f"\nTune set:")
print(f"  R²:   {r2_score(y_tune, y_tune_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_tune, y_tune_pred)):.4f}")

# evaluate on test set — only once you're happy with the model
y_test_pred = pipeline.predict(X_test)
print(f"\nTest set:")
print(f"  R²:   {r2_score(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")

NameError: name 'ColumnTransformer' is not defined